In [0]:
# ==================================================
# TSI - FASE 4: PREPARACIÓN PARA MODELADO
# PASO 55: INSPECCIÓN DE MÉTRICAS Y METADATOS
# ==================================================

from pyspark.sql import functions as F

TABLA_ORIGEN = "workspace.tsi.international_observations"

df_tsi = spark.table(TABLA_ORIGEN)

METRICAS = {
    "istanbul_traffic_index": "indice_congestion",
    "sao_paulo_cet": "congestion_longitud_m",
    "madrid_traffic_points": "flujo_veh_h",
    "metr_la": "valor_fuente",
    "pems_bay": "valor_fuente",
}

print("=== MÉTRICAS CANDIDATAS ===")

for fuente, metrica in METRICAS.items():
    print(f"{fuente}: {metrica}")

print("\n=== ESQUEMA DE COLUMNAS RELEVANTES ===")

columnas = [
    "fuente",
    "segmento_id",
    "timestamp",
    "valor_fuente",
    "velocidad_kmh",
    "flujo_veh_h",
    "ocupacion_pct",
    "indice_congestion",
    "congestion_longitud_m",
    "vmed_fuente",
    "periodo_integracion_min",
]

df_tsi.select(*columnas).printSchema()

print("\n=== MUESTRAS DE METR-LA Y PEMS-BAY ===")

for fuente in ["metr_la", "pems_bay"]:

    print(f"\nFUENTE: {fuente}")

    (
        df_tsi
        .filter(
            (F.col("fuente") == fuente)
            & (F.col("valor_fuente") > 0)
        )
        .select(
            "segmento_id",
            "timestamp",
            "valor_fuente",
            "velocidad_kmh"
        )
        .limit(5)
        .show(truncate=False)
    )

print("\n=== MUESTRAS DE MADRID ===")

(
    df_tsi
    .filter(F.col("fuente") == "madrid_traffic_points")
    .select(
        "timestamp",
        "segmento_id",
        "flujo_veh_h",
        "ocupacion_pct",
        "vmed_fuente",
        "periodo_integracion_min"
    )
    .limit(5)
    .show(truncate=False)
)

print("\nINSPECCIÓN COMPLETADA")

In [0]:
# ==================================================
# TSI - FASE 4: PREPARACIÓN PARA MODELADO
# PASO 55.2: VERIFICACIÓN DE CALIDAD METR-LA
# ==================================================

from pyspark.sql import functions as F

TABLA_CALIDAD = "workspace.tsi.metr_la_calidad"

df_calidad = spark.table(TABLA_CALIDAD)

print("=== ESTRUCTURA DE LA TABLA ===")
df_calidad.printSchema()

print("\n=== DISTRIBUCIÓN DE ETIQUETAS ===")

(
    df_calidad
    .groupBy("calidad_derivada")
    .agg(F.count("*").alias("observaciones"))
    .orderBy("calidad_derivada")
    .show(truncate=False)
)

print("\n=== VALIDACIÓN DE INTEGRIDAD ===")

total = df_calidad.count()

assert total == 7_094_304, (
    f"Total inesperado: {total}"
)

etiquetas = {
    fila["calidad_derivada"]: fila["observaciones"]
    for fila in (
        df_calidad
        .groupBy("calidad_derivada")
        .agg(F.count("*").alias("observaciones"))
        .collect()
    )
}

assert etiquetas == {
    "sospechoso_cero_global": 444_636,
    "cero_no_diagnosticado": 130_666,
    "sin_alerta_regla_qc_metr_01": 6_519_002,
}

print(f"Total validado: {total:,}")
print("Etiquetas QC-METR-01 verificadas.")

print("\n=== COLUMNAS DISPONIBLES ===")

print(df_calidad.columns)

In [0]:
# ==========================================================
# TSI - FASE 4: PREPARACIÓN PARA MODELADO
# PASO 55.3: DATASET ANALÍTICO INTERNACIONAL
# ==========================================================

from pyspark.sql import functions as F

TABLA_ORIGEN = "workspace.tsi.international_observations"
TABLA_CALIDAD = "workspace.tsi.metr_la_calidad"

df_origen = spark.table(TABLA_ORIGEN)
df_calidad = spark.table(TABLA_CALIDAD)

# 1. Seleccionar la métrica principal según la fuente.
df_modelado = (
    df_origen
    .withColumn(
        "metrica_principal",
        F.when(
            F.col("fuente") == "istanbul_traffic_index",
            F.lit("indice_congestion")
        )
        .when(
            F.col("fuente") == "sao_paulo_cet",
            F.lit("congestion_longitud_m")
        )
        .when(
            F.col("fuente") == "madrid_traffic_points",
            F.lit("flujo_veh_h")
        )
        .when(
            F.col("fuente").isin("metr_la", "pems_bay"),
            F.lit("velocidad_original")
        )
    )
    .withColumn(
        "valor_principal",
        F.when(
            F.col("fuente") == "istanbul_traffic_index",
            F.col("indice_congestion")
        )
        .when(
            F.col("fuente") == "sao_paulo_cet",
            F.col("congestion_longitud_m")
        )
        .when(
            F.col("fuente") == "madrid_traffic_points",
            F.col("flujo_veh_h")
        )
        .when(
            F.col("fuente").isin("metr_la", "pems_bay"),
            F.col("valor_fuente")
        )
    )
    .withColumn(
        "unidad_original",
        F.when(
            F.col("fuente") == "istanbul_traffic_index",
            F.lit("indice_fuente")
        )
        .when(
            F.col("fuente") == "sao_paulo_cet",
            F.lit("m")
        )
        .when(
            F.col("fuente") == "madrid_traffic_points",
            F.lit("veh/h")
        )
        .otherwise(F.lit("pendiente_confirmacion"))
    )
    .withColumn(
        "orientacion_metrica",
        F.when(
            F.col("fuente").isin(
                "istanbul_traffic_index",
                "sao_paulo_cet"
            ),
            F.lit("mayor_valor_mayor_congestion")
        )
        .when(
            F.col("fuente").isin("metr_la", "pems_bay"),
            F.lit("menor_valor_posible_deterioro")
        )
        .otherwise(F.lit("no_monotona"))
    )
)

# 2. Recuperar las etiquetas QC-METR-01.
# Se utiliza la clave sensor + timestamp para evitar
# asociar etiquetas de sensores diferentes.

df_etiquetas = (
    df_calidad
    .select(
        "timestamp",
        "segmento_id",
        "calidad_derivada"
    )
)

# Validar unicidad de la clave antes de hacer el JOIN.
duplicados = (
    df_etiquetas
    .groupBy("timestamp", "segmento_id")
    .count()
    .filter(F.col("count") > 1)
    .limit(1)
    .count()
)

assert duplicados == 0, (
    "Se detectaron claves duplicadas en METR-LA."
)

df_modelado = (
    df_modelado
    .join(
        df_etiquetas,
        on=["timestamp", "segmento_id"],
        how="left"
    )
    .select(
        "timestamp",
        "ciudad",
        "fuente",
        "segmento_id",
        "es_real",
        "metrica_principal",
        "valor_principal",
        "unidad_original",
        "orientacion_metrica",
        "calidad_derivada",
        "flujo_veh_h",
        "ocupacion_pct",
        "vmed_fuente",
        "periodo_integracion_min"
    )
)

# 3. Validar la cantidad total de observaciones.
total = df_modelado.count()

print("=== DATASET ANALÍTICO INTERNACIONAL ===")
print(f"Total de observaciones: {total:,}")

assert total == 41_770_880, (
    f"Total inesperado: {total:,}"
)

# 4. Verificar distribución por fuente.
(
    df_modelado
    .groupBy(
        "fuente",
        "metrica_principal",
        "unidad_original",
        "orientacion_metrica"
    )
    .agg(
        F.count("*").alias("observaciones"),
        F.count("valor_principal").alias("valores_validos"),
        F.count("calidad_derivada").alias("etiquetas_qc")
    )
    .orderBy("fuente")
    .show(truncate=False)
)

print("\nDataset preparado en memoria.")
print("No se han modificado las tablas originales.")

In [0]:
# ==========================================================
# TSI - FASE 4: PREPARACIÓN PARA MODELADO
# PASO 55.4: PERSISTENCIA DEL DATASET ANALÍTICO
# ==========================================================

from pyspark.sql import functions as F

TABLA_MODELO = "workspace.tsi.international_modeling"

TOTAL_ESPERADO = 41_770_880

CONTEOS_ESPERADOS = {
    "istanbul_traffic_index": 3_332,
    "sao_paulo_cet": 5_223_411,
    "madrid_traffic_points": 12_512_133,
    "metr_la": 7_094_304,
    "pems_bay": 16_937_700,
}

print("=== VALIDACIÓN PREVIA ===")

total = df_modelado.count()

assert total == TOTAL_ESPERADO, (
    f"Total inesperado: {total}"
)

conteos = {
    fila["fuente"]: fila["count"]
    for fila in df_modelado.groupBy("fuente").count().collect()
}

assert conteos == CONTEOS_ESPERADOS

print("Integridad del dataset confirmada.")

print("\n=== PERSISTENCIA EN DELTA LAKE ===")

if spark.catalog.tableExists(TABLA_MODELO):

    print("La tabla ya existe. No se realizará escritura.")

else:

    print("Creando tabla analítica...")

    (
        df_modelado
        .write
        .format("delta")
        .mode("append")
        .saveAsTable(TABLA_MODELO)
    )

    print("Tabla creada correctamente.")

print("\n=== VALIDACIÓN DE TABLA GUARDADA ===")

df_guardado = spark.table(TABLA_MODELO)

total_guardado = df_guardado.count()

assert total_guardado == TOTAL_ESPERADO

conteos_guardados = {
    fila["fuente"]: fila["count"]
    for fila in df_guardado.groupBy("fuente").count().collect()
}

assert conteos_guardados == CONTEOS_ESPERADOS

etiquetas_qc = (
    df_guardado
    .filter(F.col("fuente") == "metr_la")
    .filter(F.col("calidad_derivada").isNotNull())
    .count()
)

assert etiquetas_qc == 7_094_304

print(f"Total guardado: {total_guardado:,}")
print(f"Etiquetas QC verificadas: {etiquetas_qc:,}")

print("\n=== INFORMACIÓN DELTA ===")

(
    spark.sql(f"DESCRIBE DETAIL {TABLA_MODELO}")
    .select("format", "numFiles")
    .show(truncate=False)
)

print("\nPERSISTENCIA VALIDADA CORRECTAMENTE")
print(f"Tabla: {TABLA_MODELO}")